# Indexing MS MARCO v1 Document by OpenSearch for Sparse Encoder Model

Chunks and sparse-encodes each document server-side via the remotely hosted
`naver/splade-v3` model registered in OpenSearch — the chunk + MaxP
construction for document ranking with a passage encoder.

- [msmarco-document](https://ir-datasets.com/msmarco-document.html)
- Prerequisite: corpus downloaded via [dataset/msmarco-v1-document](../../dataset/msmarco-v1-document/README.md)
- Prerequisite: [ml_model_registration.ipynb](ml_model_registration.ipynb) with
  [model_hosting/splade-v3.py](../../model_hosting/splade-v3.py) running on the model host
  (gated HF repo, CC BY-NC-SA 4.0 — research use)

Note: splade-v3 is a **symmetric** SPLADE — queries also require a model
forward at search time (`/embed/query`).

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas opensearch-py dotenv

In [ ]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

### Index a Corpus for SPLADE Model

Encoding runs on the remote model host, so no local GPU / sentence-transformers is needed here.

In [ ]:
import ir_datasets
dataset_name = "msmarco-document"
dataset = ir_datasets.load(dataset_name)

In [ ]:
index_name = "msmarco_v1_document_splade"

In [ ]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

### Chunked Sparse Encoding (server-side ingest pipeline)

Bulk sends *raw* documents (body truncated to `MAX_DOC_CHARS`); a single
ingest pipeline runs two processors in order:

1. `text_chunking` — splits `text` into passages in `text_chunks`.
2. `sparse_encoding` — calls the remote `splade-v3` model on each passage and
   writes per-chunk `rank_features` to the nested `text_chunks_embedding` field.

In [ ]:
pipeline_id = "msmarco_v1_document_chunk_sparse"
model_id = "your-model-id"  # splade-v3 PASSAGE model (/embed/passages) from ml_model_registration

In [ ]:
def create_chunk_sparse_pipeline(
    pipeline_id: str,
    model_id: str,
    source_field: str = "text",
    chunk_field: str = "text_chunks",
    embedding_field: str = "text_chunks_embedding",
    token_limit: int = 384,
    overlap_rate: float = 0.2,
    tokenizer: str = "standard",
    batch_size: int = 64,
) -> dict:
    """
    Create (or update) an ingest pipeline that chunks then sparse-encodes text,
    fully server-side. Unlike the passage notebooks, documents are long, so the
    Robust04-style two-processor design returns:

    Stage 1 (`text_chunking`) splits `source_field` into passages in `chunk_field`.
    Stage 2 (`sparse_encoding`) embeds each passage with the remote `model_id`,
    writing a nested list of rank_features to `embedding_field`.

    `batch_size` counts DOCUMENTS, and each document carries ~2-3 chunks, so
    texts per model call ~= batch_size x avg chunks/doc. 64 (~170 chunks/call)
    is the measured sweet spot: 256 produced ~650-text calls whose long-held
    connections caused ~4% transient connection resets.
    """
    body = {
        "description": "Chunk documents, then sparse-encode each passage",
        "processors": [
            {
                "text_chunking": {
                    "algorithm": {
                        "fixed_token_length": {
                            "token_limit": token_limit,
                            "overlap_rate": overlap_rate,
                            "tokenizer": tokenizer,
                        }
                    },
                    "field_map": {source_field: chunk_field},
                }
            },
            {
                "sparse_encoding": {
                    "model_id": model_id,
                    "field_map": {chunk_field: embedding_field},
                    "batch_size": batch_size,
                }
            },
        ],
    }
    return client.ingest.put_pipeline(id=pipeline_id, body=body)

response = create_chunk_sparse_pipeline(pipeline_id, model_id, batch_size=64)
pprint.pprint(response)

Bulk indexing

In [ ]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0,
      # Disabled during bulk indexing; re-enabled after the run below.
      "refresh_interval": "-1"
    }
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "url": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
        "text_chunks": { "type": "text" },
        # Per-chunk sparse vectors produced by the sparse_encoding processor.
        # Chunking yields a list of passages, so the embeddings must be `nested`.
        "text_chunks_embedding": {
            "type": "nested",
            "properties": {
                "sparse_encoding": { "type": "rank_features" }
            }
        },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

In [ ]:
# Encoder-side truncation: keep the FIRST 10k characters (~1,700 words,
# ~2-3 chunks) of each document. Documents are never skipped — only their
# tails are dropped. For MS MARCO web documents the relevance signal
# concentrates at the start (FirstP is a strong baseline in the DL track),
# and an uncapped long tail (some docs exceed 100k chars) would explode the
# per-doc chunk count and overload the encoder host.
MAX_DOC_CHARS = 10_000

def prepare_documents(dataset):
    """
    Yield raw bulk actions. MS MARCO documents are full web pages
    (doc_id, url, title, body).
    """
    for doc in dataset.docs_iter():
        text = doc.body.replace("\n", " ")[:MAX_DOC_CHARS]
        yield {
            "_id": doc.doc_id,  # Unique identifier for the document
            "_source": {
                "docid": doc.doc_id,
                "url": doc.url,
                "title": doc.title,
                "text": text,
            }
        }

**Scale note:** 3.2M documents x ~2-3 encoded chunks each ~= the same
encoder workload as the 8.8M-passage corpus, but with longer (384-token)
sequences. Measured on a 5k benchmark (2026-08): SPLADE ~72 docs/s -> **~12.5 h**;
DPR ~34 docs/s -> **~26 h**. Both are GPU-compute-bound at 384-token chunks
(extra uvicorn workers do not help). At search time use a `nested` query with `score_mode: "max"`
(MaxP aggregation) over the chunk embeddings.

In [ ]:
from opensearchpy.helpers import parallel_bulk

# Optimized bulk settings (measured on the passage corpus, 2026-08):
# - parallel_bulk keeps batches queued at the encoder between synchronous
#   `sparse_encoding` calls
# - refresh_interval=-1 during the run (re-enabled below)
# OpenSearch 3.x removed the `_bulk?batch_size=` query param (2.x only), so we
# must NOT pass one here — it 400s the whole request.
total = dataset.docs_count()   # 3,213,835

success, errors = 0, []
with tqdm(total=total, desc="Indexing") as bar:
    for ok, item in parallel_bulk(
        client,
        prepare_documents(dataset),
        index=index_name,
        pipeline=pipeline_id,
        chunk_size=64,
        thread_count=8,
        queue_size=8,
        request_timeout=600,
        raise_on_error=False,          # collect failures instead of aborting the run
        raise_on_exception=False,
    ):
        bar.update(1)                  # advances per actually-processed document
        success += ok
        if not ok:
            errors.append(item)

print(f"indexed: {success},  failed: {len(errors)}")
if errors:
    pprint.pprint(errors[:3])          # inspect the first few errors

# Re-enable refresh now that bulk indexing is done, and make docs searchable.
client.indices.put_settings(index=index_name, body={"index": {"refresh_interval": "1s"}})
client.indices.refresh(index=index_name)
print("final count:", client.count(index=index_name)["count"])

---
### (Optional) Re-index documents missing from the first pass

If a run was interrupted, diff the corpus against what's actually in the index
and re-index just the missing ids.

At sustained multi-hour load, a small percentage of bulk items can fail with
transient `Error communicating with remote model: Connection reset`
(connection-level drops under parallel pressure; the model server itself
stays healthy). This is expected — the cells below recover exactly those
documents.

In [ ]:
from opensearchpy.helpers import scan

# All ids actually in the index (_source disabled -> fast).
indexed = set()
for hit in scan(
    client,
    index=index_name,
    query={"query": {"match_all": {}}, "_source": False},
    size=5000,
):
    indexed.add(hit["_id"])

# All ids the dataset should have produced
all_ids = {doc.doc_id for doc in dataset.docs_iter()}

missing = sorted(all_ids - indexed)
print(f"indexed: {len(indexed)},  missing: {len(missing)}")
print(missing[:10])

In [ ]:
# Re-index the documents identified as missing by the scan diff, printing every error.
from opensearchpy.helpers import parallel_bulk

def prepare_missing(dataset, missing_ids):
    docstore = dataset.docs_store()
    for doc_id in missing_ids:
        doc = docstore.get(doc_id)
        text = doc.body.replace("\n", " ")[:MAX_DOC_CHARS]
        yield {
            "_id": doc_id,
            "_source": {"docid": doc_id, "url": doc.url, "title": doc.title, "text": text},
        }

retry_ok, retry_failed = 0, []
with tqdm(total=len(missing), desc="Re-indexing") as bar:
    for ok, item in parallel_bulk(
        client,
        prepare_missing(dataset, missing),
        index=index_name,
        pipeline=pipeline_id,
        chunk_size=128,
        thread_count=4,
        queue_size=4,
        request_timeout=600,
        raise_on_error=False,
        raise_on_exception=False,
    ):
        bar.update(1)
        retry_ok += ok
        if not ok:
            retry_failed.append(item)

print(f"retried ok: {retry_ok}, still failing: {len(retry_failed)}\n")

for item in retry_failed[:10]:     # full error detail for the first failures
    pprint.pprint(item)
    print("-" * 80)